# ✈️ Flight Delay Prediction — Complete ML Pipeline
**Dataset:** US Domestic Flights 2019-2023 (3M rows)

**Target:** Binary classification — IS_DELAYED (arrival delay > 15 min)

**Scenario B:** Post-departure prediction (DEP_DELAY + TAXI_OUT known)

**Models:** HistGradientBoosting vs Random Forest

---
## Phase 1 — Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('flights_sample_3m.csv')
print(f"Shape: {df.shape}")
print(df.head())

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())

In [ ]:
df.describe()

---
## Phase 2 — Data Cleaning

In [ ]:
df_ml = df.copy()

# 1. Fill delay cause columns (NaN = no delay, not missing)
delay_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
              'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
df_ml[delay_cols] = df_ml[delay_cols].fillna(0)

# 2. Drop rows with missing core data
core_cols = ['AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'DEST',
             'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CANCELLED', 'DIVERTED',
             'DISTANCE', 'CRS_ELAPSED_TIME']
df_ml = df_ml.dropna(subset=core_cols)

# 3. Remove cancelled / diverted (no arrival = no prediction)
df_ml = df_ml[(df_ml['CANCELLED'] == 0) & (df_ml['DIVERTED'] == 0)]

# 4. Drop rows where operational columns are missing
ops_cols = ['ARR_DELAY', 'DEP_DELAY', 'TAXI_OUT', 'AIR_TIME',
            'ELAPSED_TIME', 'ARR_TIME', 'TAXI_IN', 'WHEELS_ON']
df_ml = df_ml.dropna(subset=ops_cols)

# 5. Drop redundant columns
df_ml = df_ml.drop(columns=['CANCELLATION_CODE', 'CANCELLED', 'DIVERTED',
                             'AIRLINE_DOT', 'AIRLINE'])

print(f"Clean dataset: {df_ml.shape[0]:,} rows x {df_ml.shape[1]} columns")
print(f"Remaining nulls: {df_ml.isnull().sum().sum()}")

---
## Phase 3 — Feature Engineering

In [ ]:
# === Temporal features ===
df_ml['FL_DATE'] = pd.to_datetime(df_ml['FL_DATE'])
df_ml['MONTH'] = df_ml['FL_DATE'].dt.month
df_ml['DAY_OF_WEEK'] = df_ml['FL_DATE'].dt.dayofweek
df_ml['DAY_OF_MONTH'] = df_ml['FL_DATE'].dt.day
df_ml['YEAR'] = df_ml['FL_DATE'].dt.year
df_ml['IS_WEEKEND'] = (df_ml['DAY_OF_WEEK'] >= 5).astype(int)
df_ml['DEP_HOUR'] = (df_ml['CRS_DEP_TIME'] // 100).astype(int)
df_ml['ARR_HOUR'] = (df_ml['CRS_ARR_TIME'] // 100).astype(int)

# === Season ===
df_ml['SEASON'] = df_ml['MONTH'].map({
    12:1, 1:1, 2:1, 3:2, 4:2, 5:2,
    6:3, 7:3, 8:3, 9:4, 10:4, 11:4
})

# === Time block ===
df_ml['TIME_BLOCK'] = pd.cut(df_ml['DEP_HOUR'],
    bins=[-1, 6, 12, 18, 24], labels=[0, 1, 2, 3]).astype(int)

# === Busy airport (top 20) ===
busy_airports = df_ml['ORIGIN'].value_counts().head(20).index
df_ml['IS_BUSY_ORIGIN'] = df_ml['ORIGIN'].isin(busy_airports).astype(int)

# === Clip DEP_DELAY outliers ===
low, high = df_ml['DEP_DELAY'].quantile(0.01), df_ml['DEP_DELAY'].quantile(0.99)
df_ml['DEP_DELAY'] = df_ml['DEP_DELAY'].clip(low, high)
print(f"DEP_DELAY clipped to [{low:.0f}, {high:.0f}]")

# === Target ===
df_ml['IS_DELAYED'] = (df_ml['ARR_DELAY'] > 15).astype(int)

print(f"\nTarget distribution:")
print(f"  On time (0): {(df_ml['IS_DELAYED']==0).sum():,} ({(df_ml['IS_DELAYED']==0).mean()*100:.1f}%)")
print(f"  Delayed (1): {(df_ml['IS_DELAYED']==1).sum():,} ({(df_ml['IS_DELAYED']==1).mean()*100:.1f}%)")
print(f"  Ratio: {(df_ml['IS_DELAYED']==0).sum()/(df_ml['IS_DELAYED']==1).sum():.1f}:1")

---
## Phase 4 — EDA Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

axes[0,0].hist(df_ml['ARR_DELAY'].clip(-100, 200), bins=100, color='steelblue', edgecolor='none')
axes[0,0].axvline(x=0, color='red', linestyle='--', label='On time')
axes[0,0].axvline(x=15, color='orange', linestyle='--', label='>15 min')
axes[0,0].set_title('ARR_DELAY Distribution')
axes[0,0].legend()

top_al = df_ml['AIRLINE_CODE'].value_counts().head(10).index
sns.boxplot(data=df_ml[df_ml['AIRLINE_CODE'].isin(top_al)],
            x='AIRLINE_CODE', y='ARR_DELAY', order=top_al,
            ax=axes[0,1], showfliers=False)
axes[0,1].set_title('ARR_DELAY by Airline')
axes[0,1].set_ylim(-60, 120)

monthly = df_ml.groupby('MONTH')['ARR_DELAY'].mean()
axes[0,2].bar(monthly.index, monthly.values, color='coral')
axes[0,2].set_title('Mean ARR_DELAY by Month')
axes[0,2].set_xticks(range(1,13))

hourly = df_ml.groupby('DEP_HOUR')['ARR_DELAY'].mean()
axes[1,0].plot(hourly.index, hourly.values, marker='o', color='purple')
axes[1,0].set_title('Mean ARR_DELAY by Hour')

axes[1,1].pie([(df_ml['IS_DELAYED']==0).sum(), (df_ml['IS_DELAYED']==1).sum()],
              labels=['On Time','Delayed >15min'],
              autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'])
axes[1,1].set_title('Delay Proportion')

corr_cols = ['DEP_DELAY','TAXI_OUT','TAXI_IN','AIR_TIME','CRS_ELAPSED_TIME','DISTANCE','ARR_DELAY']
sns.heatmap(df_ml[corr_cols].corr(), annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=axes[1,2], annot_kws={'size':7})
axes[1,2].set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

---
## Phase 5 — Balance Dataset + Train/Test Split
**CRITICAL:** Split BEFORE any encoding to prevent data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# Undersample majority to 70/30 ratio
delayed_rows = df_ml[df_ml['IS_DELAYED'] == 1]
ontime_rows  = df_ml[df_ml['IS_DELAYED'] == 0]

ontime_down = resample(ontime_rows,
                        n_samples=int(len(delayed_rows) * 2.3),
                        random_state=42)

df_bal = pd.concat([ontime_down, delayed_rows]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced: {df_bal.shape[0]:,} rows")
print(f"Distribution:\n{df_bal['IS_DELAYED'].value_counts(normalize=True).round(3)}")

# Features for Scenario B (post-departure)
raw_features = ['DEP_DELAY', 'TAXI_OUT', 'CRS_ELAPSED_TIME', 'DISTANCE',
                'DEP_HOUR', 'ARR_HOUR', 'MONTH', 'DAY_OF_WEEK',
                'DAY_OF_MONTH', 'IS_WEEKEND', 'SEASON', 'TIME_BLOCK',
                'IS_BUSY_ORIGIN', 'AIRLINE_CODE', 'ORIGIN', 'DEST']

X = df_bal[raw_features]
y = df_bal['IS_DELAYED']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train_raw.shape[0]:,} | Test: {X_test_raw.shape[0]:,}")

---
## Phase 6 — Encoding & Scaling (Leakage-Safe)
Target encoding is fitted on **TRAIN labels only**, then applied to both sets.

In [ ]:
from sklearn.preprocessing import StandardScaler

# === Leakage-safe target encoding ===
global_mean = y_train.mean()
smoothing = 100

def target_encode(train_col, y_tr, test_col, gmean, smooth):
    """Bayesian smoothed target encoding — fit on train only."""
    stats = y_tr.groupby(train_col).agg(['mean', 'count'])
    w = stats['count'] / (stats['count'] + smooth)
    enc = w * stats['mean'] + (1 - w) * gmean
    return train_col.map(enc).fillna(gmean), test_col.map(enc).fillna(gmean)

# Encode categoricals
tr_origin, te_origin   = target_encode(X_train_raw['ORIGIN'], y_train, X_test_raw['ORIGIN'], global_mean, smoothing)
tr_dest, te_dest       = target_encode(X_train_raw['DEST'], y_train, X_test_raw['DEST'], global_mean, smoothing)
tr_airline, te_airline  = target_encode(X_train_raw['AIRLINE_CODE'], y_train, X_test_raw['AIRLINE_CODE'], global_mean, smoothing)

# Route-level
tr_route = X_train_raw['ORIGIN'] + '_' + X_train_raw['DEST']
te_route = X_test_raw['ORIGIN'] + '_' + X_test_raw['DEST']
tr_route_enc, te_route_enc = target_encode(tr_route, y_train, te_route, global_mean, smoothing)

# === Assemble features ===
numeric  = ['DEP_DELAY', 'TAXI_OUT', 'CRS_ELAPSED_TIME', 'DISTANCE']
temporal = ['DEP_HOUR', 'ARR_HOUR', 'MONTH', 'DAY_OF_WEEK',
            'DAY_OF_MONTH', 'IS_WEEKEND', 'SEASON', 'TIME_BLOCK', 'IS_BUSY_ORIGIN']

X_train = pd.concat([
    X_train_raw[numeric + temporal].reset_index(drop=True),
    tr_origin.rename('ORIGIN_TE').reset_index(drop=True),
    tr_dest.rename('DEST_TE').reset_index(drop=True),
    tr_airline.rename('AIRLINE_TE').reset_index(drop=True),
    tr_route_enc.rename('ROUTE_TE').reset_index(drop=True),
], axis=1)

X_test = pd.concat([
    X_test_raw[numeric + temporal].reset_index(drop=True),
    te_origin.rename('ORIGIN_TE').reset_index(drop=True),
    te_dest.rename('DEST_TE').reset_index(drop=True),
    te_airline.rename('AIRLINE_TE').reset_index(drop=True),
    te_route_enc.rename('ROUTE_TE').reset_index(drop=True),
], axis=1)

y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

# === Scale numeric ===
scaler = StandardScaler()
X_train[numeric] = scaler.fit_transform(X_train[numeric])
X_test[numeric]  = scaler.transform(X_test[numeric])

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Nulls: {X_train.isnull().sum().sum()}")
print(f"\nFeatures ({X_train.shape[1]}): {X_train.columns.tolist()}")

---
## Phase 7 — Feature Correlation Check

In [ ]:
corr = X_train.corrwith(y_train).sort_values(ascending=False)
print("Feature correlations with IS_DELAYED:\n")
print(corr.round(4).to_string())

---
## Phase 8 — Model Training

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight('balanced', y_train)

# Model 1: HistGradientBoosting
hgb_clf = HistGradientBoostingClassifier(
    max_iter=500,
    max_depth=8,
    learning_rate=0.05,
    min_samples_leaf=20,
    l2_regularization=0.1,
    max_bins=255,
    random_state=42
)
hgb_clf.fit(X_train.values, y_train.values, sample_weight=sample_weights)
print("Model 1: HistGradientBoosting — trained")

# Model 2: RandomForest
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=1
)
rf_clf.fit(X_train.values, y_train.values)
print("Model 2: RandomForest — trained")

---
## Phase 9 — Model Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred_hgb  = hgb_clf.predict(X_test.values)
y_pred_rf   = rf_clf.predict(X_test.values)
y_proba_hgb = hgb_clf.predict_proba(X_test.values)[:, 1]
y_proba_rf  = rf_clf.predict_proba(X_test.values)[:, 1]

print("=" * 55)
print("MODEL 1: HIST GRADIENT BOOSTING")
print("=" * 55)
print(classification_report(y_test, y_pred_hgb, target_names=['On Time', 'Delayed']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_hgb):.4f}")

print("\n" + "=" * 55)
print("MODEL 2: RANDOM FOREST")
print("=" * 55)
print(classification_report(y_test, y_pred_rf, target_names=['On Time', 'Delayed']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

print("\n" + "=" * 55)
print("WINNER")
print("=" * 55)
auc_hgb = roc_auc_score(y_test, y_proba_hgb)
auc_rf  = roc_auc_score(y_test, y_proba_rf)
winner  = 'HistGradientBoosting' if auc_hgb >= auc_rf else 'RandomForest'
print(f"Best model: {winner} (ROC-AUC = {max(auc_hgb, auc_rf):.4f})")

---
## Phase 10 — Confusion Matrices & ROC Curves

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

cm_hgb = confusion_matrix(y_test, y_pred_hgb)
sns.heatmap(cm_hgb, annot=True, fmt=',d', cmap='Blues', ax=axes[0],
            xticklabels=['On Time','Delayed'], yticklabels=['On Time','Delayed'])
axes[0].set_title('HGB Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt=',d', cmap='Oranges', ax=axes[1],
            xticklabels=['On Time','Delayed'], yticklabels=['On Time','Delayed'])
axes[1].set_title('RF Confusion Matrix')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

RocCurveDisplay.from_estimator(hgb_clf, X_test.values, y_test, ax=axes[2], name='HGB')
RocCurveDisplay.from_estimator(rf_clf, X_test.values, y_test, ax=axes[2], name='RF')
axes[2].plot([0,1],[0,1],'k--')
axes[2].set_title('ROC Curves')

plt.tight_layout()
plt.show()

---
## Phase 11 — Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

perm_hgb = permutation_importance(hgb_clf, X_test.values, y_test.values,
                                   n_repeats=5, random_state=42, n_jobs=1)
feat_hgb = pd.Series(perm_hgb.importances_mean, index=X_train.columns).sort_values()
feat_hgb.tail(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('HGB — Top 15 Features (Permutation)')

feat_rf = pd.Series(rf_clf.feature_importances_, index=X_train.columns).sort_values()
feat_rf.tail(15).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('RF — Top 15 Features')

plt.tight_layout()
plt.show()

---
## Phase 12 — Prediction Scatter Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

np.random.seed(42)
idx = np.sort(np.random.choice(len(y_test), 1000, replace=False))

for ax, proba, name, color in [(axes[0], y_proba_hgb, 'HGB', 'steelblue'),
                                (axes[1], y_proba_rf, 'RF', 'coral')]:
    ax.scatter(range(len(idx)), y_test.iloc[idx], alpha=0.4, s=10, color='black', label='Actual')
    ax.scatter(range(len(idx)), proba[idx], alpha=0.4, s=10, color=color, label=f'{name} Prob')
    ax.axhline(y=0.5, color='red', linestyle='--', label='Threshold')
    ax.set_title(f'{name} — Probability vs Actual')
    ax.set_xlabel('Sample'); ax.set_ylabel('Probability')
    ax.legend()

plt.tight_layout()
plt.show()

---
## Phase 13 — MLflow Experiment Tracking

In [ ]:
!pip install mlflow -q

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import f1_score, accuracy_score

mlflow.set_experiment("flight_delay_final")

with mlflow.start_run(run_name="HGB_ScenarioB"):
    mlflow.log_param("model", "HistGradientBoosting")
    mlflow.log_param("max_iter", 500)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("l2_regularization", 0.1)
    mlflow.log_param("balance_method", "undersample_70_30 + sample_weight")
    mlflow.log_param("scenario", "post-departure")
    mlflow.log_param("encoding", "target_encoding_leakage_safe")
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_hgb))
    mlflow.log_metric("f1_delayed", f1_score(y_test, y_pred_hgb))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba_hgb))
    mlflow.sklearn.log_model(hgb_clf, "model")
    print("HGB logged")

with mlflow.start_run(run_name="RF_ScenarioB"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("balance_method", "undersample_70_30 + class_weight")
    mlflow.log_param("scenario", "post-departure")
    mlflow.log_param("encoding", "target_encoding_leakage_safe")
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_rf))
    mlflow.log_metric("f1_delayed", f1_score(y_test, y_pred_rf))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba_rf))
    mlflow.sklearn.log_model(rf_clf, "model")
    print("RF logged")

In [ ]:
# View experiment results
exp = mlflow.get_experiment_by_name("flight_delay_final")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])

print("=" * 70)
print("MLFLOW EXPERIMENT: flight_delay_final")
print("=" * 70)
cols = ['tags.mlflow.runName', 'params.model', 'params.balance_method',
        'metrics.accuracy', 'metrics.f1_delayed', 'metrics.roc_auc']
print(runs[cols].to_string(index=False))

---
## Phase 14 — Save Best Model

In [ ]:
import joblib

joblib.dump(hgb_clf, 'best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X_train.columns.tolist(), 'feature_names.pkl')
print("Saved: best_model.pkl, scaler.pkl, feature_names.pkl")
print(f"Features: {X_train.columns.tolist()}")

---
## Phase 15 — FastAPI Backend

In [ ]:
import os, shutil
os.makedirs('api', exist_ok=True)
for f in ['best_model.pkl', 'scaler.pkl', 'feature_names.pkl']:
    shutil.copy(f, f'api/{f}')
print(f"api/ contents: {os.listdir('api')}")

In [ ]:
app_code = '''from fastapi import FastAPI
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import joblib
import numpy as np
import pandas as pd

model = joblib.load("best_model.pkl")
scaler = joblib.load("scaler.pkl")
feature_names = joblib.load("feature_names.pkl")

app = FastAPI(title="Flight Delay Prediction API", version="2.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class FlightInput(BaseModel):
    dep_delay: float          # minutes late departing
    taxi_out: float           # taxi time in minutes
    crs_elapsed_time: float   # scheduled duration
    distance: float           # miles
    dep_hour: int             # 0-23
    arr_hour: int             # 0-23
    month: int                # 1-12
    day_of_week: int          # 0=Mon, 6=Sun
    day_of_month: int         # 1-31
    is_weekend: int           # 0 or 1
    season: int               # 1-4
    time_block: int           # 0-3
    is_busy_origin: int       # 0 or 1
    origin_te: float          # origin delay rate
    dest_te: float            # dest delay rate
    airline_te: float         # airline delay rate
    route_te: float           # route delay rate

@app.get("/")
def serve_frontend():
    return FileResponse("index.html")

@app.get("/health")
def health():
    return {"status": "healthy", "model": "HistGradientBoosting", "version": "2.0"}

@app.post("/predict")
def predict(flight: FlightInput):
    data = {
        "DEP_DELAY": flight.dep_delay,
        "TAXI_OUT": flight.taxi_out,
        "CRS_ELAPSED_TIME": flight.crs_elapsed_time,
        "DISTANCE": flight.distance,
        "DEP_HOUR": flight.dep_hour,
        "ARR_HOUR": flight.arr_hour,
        "MONTH": flight.month,
        "DAY_OF_WEEK": flight.day_of_week,
        "DAY_OF_MONTH": flight.day_of_month,
        "IS_WEEKEND": flight.is_weekend,
        "SEASON": flight.season,
        "TIME_BLOCK": flight.time_block,
        "IS_BUSY_ORIGIN": flight.is_busy_origin,
        "ORIGIN_TE": flight.origin_te,
        "DEST_TE": flight.dest_te,
        "AIRLINE_TE": flight.airline_te,
        "ROUTE_TE": flight.route_te,
    }
    df = pd.DataFrame([data])[feature_names]
    numeric_features = ["DEP_DELAY", "TAXI_OUT", "CRS_ELAPSED_TIME", "DISTANCE"]
    df[numeric_features] = scaler.transform(df[numeric_features])
    prediction = model.predict(df.values)[0]
    probability = model.predict_proba(df.values)[0]
    return {
        "prediction": "DELAYED" if prediction == 1 else "ON TIME",
        "delay_probability": round(float(probability[1]), 4),
        "on_time_probability": round(float(probability[0]), 4),
    }

@app.post("/predict/batch")
def predict_batch(flights: list[FlightInput]):
    results = [predict(f) for f in flights]
    return {"predictions": results, "count": len(results)}
'''

with open('api/app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)
print("app.py written")

In [ ]:
!pip install fastapi uvicorn -q

In [ ]:
# Direct function test (no server needed)
import sys
sys.path.insert(0, 'api')
from app import predict, FlightInput

# Test 1: Flight departed 25 min late, busy airport, summer evening
t1 = predict(FlightInput(
    dep_delay=25.0, taxi_out=18.0, crs_elapsed_time=320.0, distance=2475.0,
    dep_hour=18, arr_hour=23, month=7, day_of_week=4, day_of_month=18,
    is_weekend=0, season=3, time_block=3, is_busy_origin=1,
    origin_te=0.20, dest_te=0.21, airline_te=0.19, route_te=0.22
))

# Test 2: Flight departed on time, small airport, spring morning
t2 = predict(FlightInput(
    dep_delay=-2.0, taxi_out=10.0, crs_elapsed_time=90.0, distance=350.0,
    dep_hour=8, arr_hour=10, month=4, day_of_week=1, day_of_month=10,
    is_weekend=0, season=2, time_block=1, is_busy_origin=0,
    origin_te=0.15, dest_te=0.14, airline_te=0.16, route_te=0.13
))

print("=" * 45)
print("TEST 1: Late departure, busy, summer evening")
print(f"  Prediction:  {t1['prediction']}")
print(f"  Delay prob:  {t1['delay_probability']:.1%}")
print()
print("TEST 2: On-time departure, small airport, morning")
print(f"  Prediction:  {t2['prediction']}")
print(f"  Delay prob:  {t2['delay_probability']:.1%}")
print("=" * 45)

In [ ]:
# Start API server
import subprocess, time, requests

process = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='api'
)
time.sleep(5)

try:
    r = requests.get("http://localhost:8000/health")
    print(f"Server: {r.json()}")
    print(f"Frontend: http://localhost:8000")
    print(f"API docs: http://localhost:8000/docs")
except Exception as e:
    print(f"Error: {e}")